## Project 3: Dog Bites and Dog Run Access in NYC

This project explores whether access to dog-friendly public space is related to reported dog bite incidents in New York City.

- **Dataset(s) to be used:**  
  - DOHMH Dog Bite Data (NYC Open Data)  
    https://data.cityofnewyork.us/resource/rsgh-akpg.json  
  - NYC Parks Dog Runs (NYC Open Data)  
    https://data.cityofnewyork.us/resource/hxx3-bwgv.json  

- **Analysis question:**  
  Do boroughs with more dog runs show different dog bite patterns over time?

- **Columns that will (likely) be used:**  
  - Dog bites dataset:  
    - `dateofbite`  
    - `borough`  
    - `zipcode` 
  - Dog runs dataset:  
    - `borough`  
    - `dog_area_type` 
    - `surface`   
    - `featurestatus` 

- (If you're using multiple datasets) **Columns to be used to merge/join them:**  
  - Dog bites: `borough` (cleaned to `borough_clean`) and `year` (derived from `dateofbite`)  
  - Dog runs: `borough` (mapped from borough codes to `borough_clean`)  

- **Hypothesis:**  
  Boroughs with more dog runs may show lower or slower-growing dog bite counts over time because designated dog-friendly spaces could reduce conflict in uncontrolled public settings.

- **Why this is interesting**
Dog runs are a practical, neighborhood-level policy tool. If patterns suggest a relationship with bite incidents, it could inform how the city invests in safe dog-friendly infrastructure.

In [46]:
import pandas as pd
import numpy as np
import plotly.express as px
import json

In [47]:
DOG_BITES_URL = "https://data.cityofnewyork.us/resource/rsgh-akpg.csv"
DOG_RUNS_URL = "https://data.cityofnewyork.us/resource/hxx3-bwgv.csv"

# Pull a large but manageable amount
dog_bites = pd.read_csv(f"{DOG_BITES_URL}?$limit=50000")
dog_runs = pd.read_csv(f"{DOG_RUNS_URL}?$limit=5000")

print("Dog bites shape:", dog_bites.shape)
print("Dog runs shape:", dog_runs.shape)


Dog bites shape: (34033, 9)
Dog runs shape: (91, 16)


In [48]:
dog_bites.head()

,uniqueid,dateofbite,species,breed,age,gender,spayneuter,borough,zipcode
0,1,2018-01-01T00:00:00.000,DOG,UNKNOWN,NaN,U,False,Brooklyn,11220
1,2,2018-01-04T00:00:00.000,DOG,UNKNOWN,NaN,U,False,Brooklyn,NaN
2,3,2018-01-06T00:00:00.000,DOG,Pit Bull,NaN,U,False,Brooklyn,11224
3,4,2018-01-08T00:00:00.000,DOG,Mixed/Other,4,M,False,Brooklyn,11231
4,5,2018-01-09T00:00:00.000,DOG,Pit Bull,NaN,U,False,Brooklyn,11224


In [49]:
dog_runs.head()

,objectid,system,the_geom,gispropnum,department,parentid,communityboard,councildistrict,precinct,zipcode,borough,dog_area_type,name,seating,surface,featurestatus
0,2797,B082-DOGAREA0045,MULTIPOLYGON (((-74.03650678412465 40.61231389...,B082,B-10,B082,310,43,68,11209,B,Dog Run,Frank Decolvenaere Dog Run,Yes,Synthetic,Active
1,256,X110A-DOGAREA0022,MULTIPOLYGON (((-73.905665046473 40.9013570516...,X110A,X-08,X110A,208,11,50,10471,X,Dog Run,Frank S. Hackett Park Dog Run,NaN,NaN,Active
2,2789,B068-DOGAREA0044,MULTIPOLYGON (((-73.9714968265382 40.649229616...,B068,B-14,B068,314,40,70,11226,B,Dog Run,Kensington Dog Run,Yes,Natural,Active
3,2788,Q066C-DOGAREA0018,MULTIPOLYGON (((-73.92203493112896 40.77402156...,Q066C,Q-01,Q066C,401,22,114,11102,Q,Dog Run,Triborough Bridge Playground C Dog Run,Yes,Asphalt,Active
4,2388,X002-DOGAREA0029,MULTIPOLYGON (((-73.8707908532998 40.855740791...,X002,X-14,X002,227,15,49,10462,X,Dog Run,Bronx River Park Dog Run,Yes,NaN,Active


In [50]:
print("Dog bite columns:")
print(dog_bites.columns.tolist())

print("\nDog run columns:")
print(dog_runs.columns.tolist())

Dog bite columns:
['uniqueid', 'dateofbite', 'species', 'breed', 'age', 'gender', 'spayneuter', 'borough', 'zipcode']

Dog run columns:
['objectid', 'system', 'the_geom', 'gispropnum', 'department', 'parentid', 'communityboard', 'councildistrict', 'precinct', 'zipcode', 'borough', 'dog_area_type', 'name', 'seating', 'surface', 'featurestatus']


### Cleaning and preparing dog bite data

Here, I will parse the bite date, create a year column and standardize borough names

In [51]:

DOG_DATE_COL = "dateofbite"
DOG_BORO_COL = "borough"

dog_bites[DOG_DATE_COL] = pd.to_datetime(dog_bites[DOG_DATE_COL], errors="coerce")
dog_bites = dog_bites.dropna(subset=[DOG_DATE_COL, DOG_BORO_COL]).copy()

dog_bites["year"] = dog_bites[DOG_DATE_COL].dt.year
dog_bites["borough_clean"] = dog_bites[DOG_BORO_COL].astype(str).str.strip().str.title()

dog_bites[["borough_clean", "year"]].head(10)


,borough_clean,year
0,Brooklyn,2018
1,Brooklyn,2018
2,Brooklyn,2018
3,Brooklyn,2018
4,Brooklyn,2018
5,Brooklyn,2018
6,Brooklyn,2018
7,Brooklyn,2018
8,Brooklyn,2018
9,Brooklyn,2018


### Cleaning and preparing dog runs data

Here, I standardize borough names and count dog runs per borough.


In [52]:
RUNS_BORO_COL = "borough"

boro_code_map = {
    "B": "Brooklyn",
    "M": "Manhattan",
    "Q": "Queens",
    "R": "Staten Island",
    "X": "Bronx"
}

# Standardizing raw borough codes and map to full names
dog_runs["borough_code"] = dog_runs[RUNS_BORO_COL].astype(str).str.strip().str.upper()

dog_runs["borough_clean"] = dog_runs["borough_code"].map(boro_code_map)

# Counting dog runs per borough
dog_runs_counts = (
    dog_runs.dropna(subset=["borough_clean"])
    .groupby("borough_clean", as_index=False)
    .size()
    .rename(columns={"size": "dog_runs_count"})
)

dog_runs_counts


,borough_clean,dog_runs_count
0,Bronx,14
1,Brooklyn,20
2,Manhattan,39
3,Queens,13
4,Staten Island,5


### Aggregate dog bites by borough and year


In [53]:
bites_by_boro_year = (
    dog_bites
    .dropna(subset=["year"])
    .groupby(["borough_clean", "year"], as_index=False)
    .size()
    .rename(columns={"size": "dog_bites_count"})
)

bites_by_boro_year.head()


,borough_clean,year,dog_bites_count
0,Bronx,2015,595
1,Bronx,2016,576
2,Bronx,2017,586
3,Bronx,2018,599
4,Bronx,2019,611


### Merging dog run access with dog bite trends

Now I merge borough-level dog run counts onto yearly dog bite totals.


In [54]:
merged = pd.merge(
    bites_by_boro_year,
    dog_runs_counts,
    on="borough_clean",
    how="left"
)

merged.head()


,borough_clean,year,dog_bites_count,dog_runs_count
0,Bronx,2015,595,14.0
1,Bronx,2016,576,14.0
2,Bronx,2017,586,14.0
3,Bronx,2018,599,14.0
4,Bronx,2019,611,14.0


### Visualization: Dog bites over time by borough


In [55]:
fig = px.line(
    bites_by_boro_year,
    x="year",
    y="dog_bites_count",
    color="borough_clean",
    title="Reported Dog Bites Over Time by Borough"
)
fig.show()



Across boroughs, reported dog bites appear to dip around 2020 and then rise again afterward.  
This pattern may reflect changes in daily routines, reporting behavior, or dog ownership trends during and after the pandemic.

Boroughs also differ in baseline level. Even before considering dog run access, the borough lines suggest that dog bites are not evenly distributed across the city.


### Visualization: Dog run access by borough


In [56]:
fig = px.bar(
    dog_runs_counts,
    x="borough_clean",
    y="dog_runs_count",
    title="Dog Runs Count by Borough"
)
fig.show()


Manhattan appears to have the largest number of dog runs in this dataset, while Staten Island has the fewest.

This is a simple count, not a per-capita measure, but it gives a useful first snapshot of how dog-friendly infrastructure is distributed at a borough level.

### Visualization: Dog runs vs average dog bites

This gives a simple borough-level relationship view.


In [57]:
avg_bites = (
    bites_by_boro_year
    .groupby("borough_clean", as_index=False)
    .agg(avg_dog_bites=("dog_bites_count", "mean"))
)

scatter_df = pd.merge(avg_bites, dog_runs_counts, on="borough_clean", how="left")

fig = px.scatter(
    scatter_df,
    x="dog_runs_count",
    y="avg_dog_bites",
    text="borough_clean",
    title="Dog Runs vs Average Reported Dog Bites (Borough)"
)
fig.update_traces(textposition="top center")
fig.show()

scatter_df


,borough_clean,avg_dog_bites,dog_runs_count
0,Bronx,602.5,14.0
1,Brooklyn,731.7,20.0
2,Manhattan,769.4,39.0
3,Other,148.2,NaN
4,Queens,865.2,13.0
5,Staten Island,286.3,5.0


At a borough level, the relationship between dog runs and average reported bites looks mixed rather than clearly linear. Manhattan has both high dog run counts and relatively high average bite counts, while Staten Island shows low dog run counts and low average bite counts.

This suggests that dog runs alone do not explain borough-level bite differences. Other factors like population size, dog ownership rates, reporting behavior, and neighborhood density likely matter.

### Dog run surface by borough

I examine whether dog run **surface** varies across boroughs.
This gives a more meaningful sense of dog run quality and design differences.


In [58]:
dog_runs_surface_counts = (
    dog_runs.dropna(subset=["borough_clean", "surface"])
    .groupby(["borough_clean", "surface"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

dog_runs_surface_counts.head()



,borough_clean,surface,count
0,Bronx,Concrete,1
1,Bronx,Natural,2
2,Bronx,Sand,1
3,Brooklyn,Natural,2
4,Brooklyn,Sand,2


In [59]:
fig = px.bar(
    dog_runs_surface_counts,
    x="borough_clean",
    y="count",
    color="surface",
    title="Dog Run Surface by Borough"
)
fig.show()



## Conclusion

My hypothesis was that boroughs with more dog runs might show lower or slower-growing dog bite counts over time because designated off-leash spaces could reduce conflict in uncontrolled settings.

### Main pattern in dog bites over time
Across boroughs, reported dog bites appear relatively steady through the mid-to-late 2010s, followed by a clear dip around 2020 and then a rebound in the years after. This pattern is visible across multiple borough lines, suggesting a citywide shift rather than a single-borough effect.

### Dog runs and bite levels
When I compare average dog bites to dog run counts by borough, I do not see a simple relationship. Manhattan has the highest number of dog runs but does not have the lowest average bites. Queens appears to have relatively high average bites with fewer dog runs than Manhattan. Staten Island has both fewer dog runs and lower average bite counts. Overall, the pattern looks mixed, so dog run count alone does not seem to explain borough-level bite differences.

### Limitations
This analysis has several limitations:
1. I used dog run counts as a proxy for access to dog-friendly public space, but this does not capture the size, quality, or usage of each dog run.  
2. The dog bite data likely reflects reporting behavior, which can vary by borough.  
3. Borough population size, dog ownership rates, and neighborhood density are not controlled for here. 
4. This is a borough-level view, so it may hide meaningful neighborhood-level patterns.

### Note on the use of AI tools

I used ChatGPT to help clarify my research question, sanity check my approach to cleaning and aggregating the datasets,  and refine the wording of my markdown so the notebook reads as a coherent narrative.